# Temporal distributions in the background system

This notebook shows a feature that complements the [Getting Started](getting_started.ipynb) example: temporal distributions defined **inside background databases**.

Normally, `bw_timex` stops the temporal graph traversal at the *first* background process and treats everything behind it as a static snapshot. With the optional flag `traverse_background=True`, the traversal descends **into** the background, so that a `TemporalDistribution` on a background exchange also spreads its downstream flows over time — and each time-shifted flow is sourced from the temporally-appropriate background database variant.

We'll build a tiny decarbonization story to see why this matters.

## The system

Our foreground process `A` buys a background component `B`. Producing `B` consumes electricity `C`, and electricity causes CO₂. Both `B` and `C` live in the background.

```mermaid
flowchart LR
subgraph background[<i>background</i>]
    B("Component B"):::bg
    C("Electricity C"):::bg
end

subgraph foreground[<i>foreground</i>]
    A("Process A"):::fg
end

subgraph biosphere[<i>biosphere</i>]
    CO2("CO2"):::bio
end

B-->|"3 kg"|A
C-->|"2 kWh"|B
C-.->|"11 kg"|CO2

classDef fg color:#222832, fill:#3fb1c5, stroke:none;
classDef bg color:#222832, fill:#f4a259, stroke:none;
classDef bio color:#222832, fill:#9c5ffd, stroke:none;
style background fill:none, stroke:none;
style foreground fill:none, stroke:none;
style biosphere fill:none, stroke:none;
```

In [1]:
import bw2data as bd

bd.projects.set_current("background_temporalization")

bd.Database("biosphere").write(
    {
        ("biosphere", "CO2"): {"type": "emission", "name": "CO2"},
    }
)

bd.Database("background_2020").write(
    {
        ("background_2020", "B"): {
            "name": "B",
            "location": "somewhere",
            "reference product": "B",
            "exchanges": [
                {"amount": 1, "type": "production", "input": ("background_2020", "B")},
                {"amount": 2, "type": "technosphere", "input": ("background_2020", "C")},
            ],
        },
        ("background_2020", "C"): {
            "name": "C",
            "location": "somewhere",
            "reference product": "C",
            "exchanges": [
                {"amount": 1, "type": "production", "input": ("background_2020", "C")},
                {"amount": 11, "type": "biosphere", "input": ("biosphere", "CO2")},
            ],
        },
    }
)

bd.Database("foreground").write(
    {
        ("foreground", "A"): {
            "name": "A",
            "location": "somewhere",
            "reference product": "A",
            "exchanges": [
                {"amount": 1, "type": "production", "input": ("foreground", "A")},
                {"amount": 3, "type": "technosphere", "input": ("background_2020", "B")},
            ],
        },
    }
)

bd.Method(("our", "method")).write([(("biosphere", "CO2"), 1)])

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 10754.63it/s]

10:08:48+0200 [info     ] Vacuuming database            


10:08:48+0200 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


  0%|          | 0/2 [00:00<?, ?it/s]

...

10:08:48+0200 [info     ] Vacuuming database            


10:08:48+0200 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 10866.07it/s]

10:08:48+0200 [info     ] Vacuuming database            


### A decarbonizing grid

Our background represents the year 2020, where electricity `C` emits 11 kg CO₂. By 2040 the grid is cleaner, so the same electricity only emits 7 kg CO₂. We write this as a separate, prospective background database.

```mermaid
flowchart LR
subgraph background[<i>background</i>]
    C2020("Electricity C \n 2020"):::bg
    C2040("Electricity C \n 2040"):::bg
end

subgraph biosphere[<i>biosphere</i>]
    CO2("CO2"):::bio
end

C2020-.->|"<span style='color:#9c5ffd'><b>11 kg</b></span>"|CO2
C2040-.->|"<span style='color:#9c5ffd'><b>7 kg</b></span>"|CO2

classDef bg color:#222832, fill:#f4a259, stroke:none;
classDef bio color:#222832, fill:#9c5ffd, stroke:none;
style background fill:none, stroke:none;
style biosphere fill:none, stroke:none;
```

In [2]:
bd.Database("background_2040").write(
    {
        ("background_2040", "B"): {
            "name": "B",
            "location": "somewhere",
            "reference product": "B",
            "exchanges": [
                {"amount": 1, "type": "production", "input": ("background_2040", "B")},
                {"amount": 2, "type": "technosphere", "input": ("background_2040", "C")},
            ],
        },
        ("background_2040", "C"): {
            "name": "C",
            "location": "somewhere",
            "reference product": "C",
            "exchanges": [
                {"amount": 1, "type": "production", "input": ("background_2040", "C")},
                {"amount": 7, "type": "biosphere", "input": ("biosphere", "CO2")},
            ],
        },
    }
)

for db in bd.databases:
    bd.Database(db).process()

10:08:48+0200 [warning  ] Not able to determine geocollections for all datasets. This database is not ready for regionalization.


  0%|          | 0/2 [00:00<?, ?it/s]

100%|██████████| 2/2 [00:00<00:00, 48489.06it/s]

10:08:48+0200 [info     ] Vacuuming database            


As in Getting Started, we record which background database represents which point in time:

In [3]:
from datetime import datetime

database_dates = {
    "background_2020": datetime.strptime("2020", "%Y"),
    "background_2040": datetime.strptime("2040", "%Y"),
    "foreground": "dynamic",
}

## A temporal distribution on a *background* exchange

Here's the new part. Component `B` doesn't use its electricity all at once: 60% is used right away, and 40% ten years later (think of a long-lived component being operated over its lifetime). We attach this `TemporalDistribution` to the **background** exchange `C → B` — exactly the same way we would for a foreground exchange.

```mermaid
flowchart LR
subgraph background[<i>background</i>]
    B("Component B"):::bg
    C("Electricity C"):::bg
end

subgraph foreground[<i>foreground</i>]
    A("Process A"):::fg
end

B-->|"3 kg"|A
C-->|"amounts: [60%, 40%] * 2 kWh<br/> dates: [0, +10]" years|B

classDef fg color:#222832, fill:#3fb1c5, stroke:none;
classDef bg color:#222832, fill:#f4a259, stroke:none;
style background fill:none, stroke:none;
style foreground fill:none, stroke:none;
```

We add the same distribution to the `C → B` exchange in **both** background variants, so the temporal behaviour is defined consistently for each point in time.

In [4]:
import numpy as np
from bw_temporalis import TemporalDistribution
from bw_timex.utils import add_temporal_distribution_to_exchange

td_c_to_b = TemporalDistribution(
    date=np.array([0, 10], dtype="timedelta64[Y]"),
    amount=np.array([0.6, 0.4]),
)

for background in ["background_2020", "background_2040"]:
    add_temporal_distribution_to_exchange(
        temporal_distribution=td_c_to_b,
        input_code="C",
        input_database=background,
        output_code="B",
        output_database=background,
    )

2026-06-22 10:08:49.308 | INFO     | bw_timex.utils:add_temporal_distribution_to_exchange:604 - Added temporal distribution to exchange Exchange: 2 None 'C' (None, somewhere, None) to 'B' (None, somewhere, None).


2026-06-22 10:08:49.311 | INFO     | bw_timex.utils:add_temporal_distribution_to_exchange:604 - Added temporal distribution to exchange Exchange: 2 None 'C' (None, somewhere, None) to 'B' (None, somewhere, None).


## Without background traversal (the default)

By default, `bw_timex` stops at the first background process (`B`) and treats its supply chain as a static snapshot. The component `B` is sourced from a time-interpolated mix of the 2020 and 2040 databases (because the demand occurs in 2024), but the temporal distribution *inside* `B` (`C → B`) is **not** honoured — the background is static.

In [5]:
from bw_timex import TimexLCA

tlca_static_bg = TimexLCA(
    demand={("foreground", "A"): 1},
    method=("our", "method"),
    database_dates=database_dates,
)
tlca_static_bg.build_timeline(starting_datetime="2024-01-01")
tlca_static_bg.timeline

2026-06-22 10:08:49.315 | INFO     | bw_timex.timex_lca:__init__:136 - Initializing TimexLCA object...


2026-06-22 10:08:49.316 | INFO     | bw_timex.timex_lca:__init__:153 - Calculating base LCA...


2026-06-22 10:08:49.331 | INFO     | bw_timex.timex_lca:__init__:170 - Collecting node infos...


2026-06-22 10:08:49.332 | INFO     | bw_timex.timex_lca:build_timeline:336 - No edge filter function provided. Skipping all edges in background databases.


2026-06-22 10:08:49.333 | INFO     | bw_timex.timex_lca:build_timeline:357 - Creating activity time mapping...


2026-06-22 10:08:49.333 | INFO     | bw_timex.timeline_builder:__init__:112 - Traversing supply chain graph...


2026-06-22 10:08:49.336 | INFO     | bw_timex.timeline_builder:build_timeline:186 - Building timeline...


Starting graph traversal
Calculation count: 1


,hash_producer,time_mapped_producer,date_producer,producer,producer_name,hash_consumer,time_mapped_consumer,date_consumer,consumer,consumer_name,amount,temporal_market_shares,temporal_evolution,temporal_evolution_reference
0,2024,327359257374097410,2024-01-01,327359257227296768,B,2024,327359257374097411,2024-01-01,327359257286017024,A,3.0,"{'background_2020': 0.8, 'background_2040': 0.2}",None,producer
1,2024,327359257374097411,2024-01-01,327359257286017024,A,2024,-1,2024-01-01,-1,-1,1.0,None,None,producer


The timeline stops at `B`: it is a single "temporal market" in 2024, split between the two background variants. We never see electricity `C` in the timeline.

In [6]:
tlca_static_bg.lci()
tlca_static_bg.static_lcia()
tlca_static_bg.static_score

2026-06-22 10:08:49.359 | INFO     | bw_timex.timex_lca:lci:507 - Expanding matrices...


2026-06-22 10:08:49.361 | INFO     | bw_timex.timex_lca:lci:526 - Calculating dynamic inventory...


61.20000000000002

## With `traverse_background=True`

Now we switch on background traversal. The only change is one keyword argument to `build_timeline`.

In [7]:
tlca_dynamic_bg = TimexLCA(
    demand={("foreground", "A"): 1},
    method=("our", "method"),
    database_dates=database_dates,
)
tlca_dynamic_bg.build_timeline(
    starting_datetime="2024-01-01",
    traverse_background=True,
)

2026-06-22 10:08:49.371 | INFO     | bw_timex.timex_lca:__init__:136 - Initializing TimexLCA object...


2026-06-22 10:08:49.372 | INFO     | bw_timex.timex_lca:__init__:153 - Calculating base LCA...


2026-06-22 10:08:49.378 | INFO     | bw_timex.timex_lca:__init__:170 - Collecting node infos...


2026-06-22 10:08:49.379 | WARNING  | bw_timex.timex_lca:build_timeline:303 - traverse_background=True with graph_traversal='priority': non-referenced background variants are not placed on the priority heap; each variant subtree is walked in full via proxy reads when its parent edge is reached. The referenced-system heap exploration order is unchanged and explored amounts are exact (identical to graph_traversal='bfs' for these subtrees).


2026-06-22 10:08:49.380 | INFO     | bw_timex.timex_lca:build_timeline:357 - Creating activity time mapping...


2026-06-22 10:08:49.380 | INFO     | bw_timex.timeline_builder:__init__:112 - Traversing supply chain graph...


2026-06-22 10:08:49.384 | INFO     | bw_timex.timeline_builder:build_timeline:186 - Building timeline...


Starting graph traversal
Calculation count: 2


,date_producer,producer_name,date_consumer,consumer_name,amount,temporal_market_shares
0,2024-01-01,B,2024-01-01,A,3.0,None
1,2024-01-01,C,2024-01-01,B,1.2,"{'background_2020': 0.8, 'background_2040': 0.2}"
2,2024-01-01,A,2024-01-01,-1,1.0,None
3,2034-01-01,C,2024-01-01,B,0.8,"{'background_2020': 0.3, 'background_2040': 0.7}"


Now the traversal descends into `B` and the `C → B` temporal distribution takes effect. Electricity `C` shows up **twice**, and each cohort is sourced from a time-interpolated grid mix:

- 60% of it in **2024** — mostly the 2020 grid (`{'background_2020': 0.8, 'background_2040': 0.2}`), and
- 40% of it in **2034** — mostly the cleaner 2040 grid (`{'background_2020': 0.3, 'background_2040': 0.7}`).

Because the later electricity leans on the future, cleaner grid, the total impact is lower:

In [8]:
tlca_dynamic_bg.lci()
tlca_dynamic_bg.static_lcia()
tlca_dynamic_bg.static_score

2026-06-22 10:08:49.401 | INFO     | bw_timex.timex_lca:lci:507 - Expanding matrices...


2026-06-22 10:08:49.405 | INFO     | bw_timex.timex_lca:lci:526 - Calculating dynamic inventory...


56.39999999999999

## Comparison

| | CO₂ score |
|---|---|
| Plain static LCA (2020 grid only) | 66.0 |
| `TimexLCA`, static background | 61.2 |
| `TimexLCA`, `traverse_background=True` | 56.4 |

We can check the `traverse_background=True` number by hand. The demand pulls `3 × 2 = 6` kWh of electricity, split 60/40 by the temporal distribution, with each cohort interpolated linearly between the 2020 and 2040 grids:

- **3.6 kWh in 2024** → `0.8 × 11 + 0.2 × 7 = 10.2` kg/kWh → `36.72` kg
- **2.4 kWh in 2034** → `0.3 × 11 + 0.7 × 7 = 8.2` kg/kWh → `19.68` kg

Total: `56.4` kg CO₂. ✅

> **Note:** This works identically with `graph_traversal="bfs"`. In the default `"priority"` mode a one-time informational warning is emitted, because non-referenced background variants are explored via direct database reads rather than through the priority heap — the explored amounts are exact either way.

## Composing foreground and background temporal distributions

Background distributions compose with foreground ones. Suppose `A` also doesn't buy component `B` all at once — 70% now and 30% in six years. We add a `TemporalDistribution` to the foreground edge `B → A` (just like in [Getting Started](getting_started.ipynb)) and rebuild with `traverse_background=True`. Now **both** edges carry temporal information:

```mermaid
flowchart LR
subgraph background[<i>background</i>]
    B("Component B"):::bg
    C("Electricity C"):::bg
end

subgraph foreground[<i>foreground</i>]
    A("Process A"):::fg
end

B-->|"amounts: [70%, 30%] * 3 kg<br/> dates: [0, +6]" years|A
C-->|"amounts: [60%, 40%] * 2 kWh<br/> dates: [0, +10]" years|B

classDef fg color:#222832, fill:#3fb1c5, stroke:none;
classDef bg color:#222832, fill:#f4a259, stroke:none;
style background fill:none, stroke:none;
style foreground fill:none, stroke:none;
```

In [9]:
add_temporal_distribution_to_exchange(
    temporal_distribution=TemporalDistribution(
        date=np.array([0, 6], dtype="timedelta64[Y]"),
        amount=np.array([0.7, 0.3]),
    ),
    input_code="B",
    input_database="background_2020",
    output_code="A",
    output_database="foreground",
)

tlca_both = TimexLCA(
    demand={("foreground", "A"): 1},
    method=("our", "method"),
    database_dates=database_dates,
)
tlca_both.build_timeline(starting_datetime="2024-01-01", traverse_background=True)
tlca_both.timeline

2026-06-22 10:08:49.420 | INFO     | bw_timex.utils:add_temporal_distribution_to_exchange:604 - Added temporal distribution to exchange Exchange: 3 None 'B' (None, somewhere, None) to 'A' (None, somewhere, None).


2026-06-22 10:08:49.420 | INFO     | bw_timex.timex_lca:__init__:136 - Initializing TimexLCA object...


2026-06-22 10:08:49.420 | INFO     | bw_timex.timex_lca:__init__:153 - Calculating base LCA...


2026-06-22 10:08:49.429 | INFO     | bw_timex.timex_lca:__init__:170 - Collecting node infos...


2026-06-22 10:08:49.430 | WARNING  | bw_timex.timex_lca:build_timeline:303 - traverse_background=True with graph_traversal='priority': non-referenced background variants are not placed on the priority heap; each variant subtree is walked in full via proxy reads when its parent edge is reached. The referenced-system heap exploration order is unchanged and explored amounts are exact (identical to graph_traversal='bfs' for these subtrees).


2026-06-22 10:08:49.431 | INFO     | bw_timex.timex_lca:build_timeline:357 - Creating activity time mapping...


2026-06-22 10:08:49.431 | INFO     | bw_timex.timeline_builder:__init__:112 - Traversing supply chain graph...


2026-06-22 10:08:49.435 | INFO     | bw_timex.timeline_builder:build_timeline:186 - Building timeline...


Starting graph traversal
Calculation count: 2


,hash_producer,time_mapped_producer,date_producer,producer,producer_name,hash_consumer,time_mapped_consumer,date_consumer,consumer,consumer_name,amount,temporal_market_shares,temporal_evolution,temporal_evolution_reference
0,2024,327359257374097410,2024-01-01,327359257227296768,B,2024,327359257374097412,2024-01-01,327359257286017024,A,2.1,None,None,producer
1,2024,327359257374097411,2024-01-01,327359257227296769,C,2024,327359257374097410,2024-01-01,327359257227296768,B,1.2,"{'background_2020': 0.8, 'background_2040': 0.2}",None,producer
2,2024,327359257374097412,2024-01-01,327359257286017024,A,2024,-1,2024-01-01,-1,-1,1.0,None,None,producer
3,2030,327359257374097413,2030-01-01,327359257227296768,B,2024,327359257374097412,2024-01-01,327359257286017024,A,0.9,None,None,producer
4,2030,327359257374097414,2030-01-01,327359257227296769,C,2030,327359257374097413,2030-01-01,327359257227296768,B,1.2,"{'background_2020': 0.5, 'background_2040': 0.5}",None,producer
5,2034,327359257374097415,2034-01-01,327359257227296769,C,2024,327359257374097410,2024-01-01,327359257227296768,B,0.8,"{'background_2020': 0.3, 'background_2040': 0.7}",None,producer
6,2040,327359257374097416,2040-01-01,327359257227296769,C,2030,327359257374097413,2030-01-01,327359257227296768,B,0.8,{'background_2040': 1},None,producer


The foreground distribution (*when* `B` is bought) convolves with the background distribution (*when* `B` uses electricity). Component `B` now appears in **2024** and **2030**, and electricity `C` fans out into **four** cohorts — 2024, 2030, 2034 and 2040 — each routed to the appropriate grid. The score drops further:

In [10]:
tlca_both.lci()
tlca_both.static_lcia()
tlca_both.static_score

2026-06-22 10:08:49.452 | INFO     | bw_timex.timex_lca:lci:507 - Expanding matrices...


2026-06-22 10:08:49.456 | INFO     | bw_timex.timex_lca:lci:526 - Calculating dynamic inventory...


54.239999999999995

## Recap

Temporalizing the background is just two things:

1. Put a `TemporalDistribution` on a background exchange (in each time-specific background database).
2. Pass `traverse_background=True` to `build_timeline`.

Foreground and background temporal distributions compose automatically through the supply-chain convolution, and everything else — `lci()`, `static_lcia()`, dynamic characterization — works exactly as in [Getting Started](getting_started.ipynb).